In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    '''one head of self-attention'''

    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd,head_size,bias = False)
        self.query = nn.Linear(n_embd,head_size,bias = False)
        self.value = nn.Linear(n_embd,head_size,bias = False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)     #(B,T,C)
        q = self.query(x)      #(B,T,C)
        #compute attention scores("affinities")
        wei = q @ k.transpose(-2,-1)*C**-0.5  #(B,T,C) @ (B,C,T) ---->(B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0,float('-inf')) #(B,T,T)
        wei = F.softmax(wei,dim = -1)   #(B,T,T)
        wei = self.dropout(wei)

        v = self.value(x)   #(B,T,C)
        out = wei @ v   #(B,T,T) @ (B,T,C) ---->(B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    '''multiple heads of self-attention in parallel'''
    
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd,n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads],dim = -1)
        out = self.proj(self.dropout(out))
        return out

class FeedForward(nn.Module):
    '''a simple linear layer followed by a non-linearity'''

    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd),
            nn.Dropout(dropout),
        )

    def forward(self,x):
        return self.net(x)

class LayerNorm(nn.Module):
    def __init__(self,dim,eps = 1e-5):
        self.eps = eps
        #parameters(trained with backprop)通过反向传播进行训练/更新
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self,x):
        #calculate the forward pass
        xmean = x.mean(1,keepdim = True) #batch mean
        xvar = x.var(1,keepdim = True) #batch variance
        xhat = (x-xmean)/torch.sqrt(xvar+self.eps)
        out = self.gamma * xhat + self.beta
        return out

class Block(nn.Module):
    '''Transformer block:communication followed by computation'''

    def __init__(self,n_embd,n_head):
        #n_embd:embedding dimension,n_head:the number of heads we'd like
        super().__init__()
        head_size = n_embd//n_head
        self.sa = MultiHeadAttention(n_head,head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        #self.sa_heads = MultiHeadAttention(4,n_embd//4)    #4 heads of 8-dimensional self-attention
        #self.ffwd = FeedForward(n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd,n_head = n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)    #final layer norm
        self.lm_head = nn.Linear(n_embd,vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape

         # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)   # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T,device = device))  #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        #x = self.sa_heads(x)  #apply one head of self-attention. (B,T,C)
        #x = self.ffwd(x)    #(B,T,C)
        x = self.blocks(x)  # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #crop idx to the last block_size tokens
            idx_cond = idx[:,-block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


step 0: train loss 4.2227, val loss 4.2226
step 300: train loss 2.8402, val loss 2.8482
step 600: train loss 2.6071, val loss 2.6175
step 900: train loss 2.5209, val loss 2.5235
step 1200: train loss 2.4699, val loss 2.4689
step 1500: train loss 2.4201, val loss 2.4294
step 1800: train loss 2.3949, val loss 2.4017
step 2100: train loss 2.3656, val loss 2.3681
step 2400: train loss 2.3427, val loss 2.3595
step 2700: train loss 2.3246, val loss 2.3454
step 3000: train loss 2.3231, val loss 2.3249
step 3300: train loss 2.3080, val loss 2.3153
step 3600: train loss 2.2897, val loss 2.3091
step 3900: train loss 2.2832, val loss 2.3017
step 4200: train loss 2.2751, val loss 2.3063
step 4500: train loss 2.2613, val loss 2.2749
step 4800: train loss 2.2517, val loss 2.2729

-be trow, ang rop alrame,
Sorsolts rea whoutoo-
GBin! my ranmy thlay thur an efies; ne shat theakedce singhaill dre sm lulis my whip I me wall-Vingth my marker his atst bus fhave 'Buts, com ef cpnere. KELARIICENTIRIOLELIORO

```python
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd,vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)   # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T,device = device))  #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
```
![](images/2026-10-19-15-17-35.png)
![](images/2026-10-19-15-21-48.png)
![](images/2026-10-19-16-24-39.png)
![](images/2026-10-19-16-24-59.png)
![](images/2026-10-19-16-25-15.png)
![](images/2026-10-19-16-26-01.png)
![](images/2026-10-19-16-45-42.png)
![](images/2026-10-19-16-47-01.png)
![](images/2026-10-19-16-55-39.png)
![](images/2026-10-19-16-59-15.png)
![](images/2026-10-19-17-07-21.png)
![](images/2026-10-19-16-59-27.png)
![](images/2026-10-19-17-01-18.png)
![](images/2026-10-19-17-02-29.png)
![](images/2026-10-19-17-45-15.png)
![](images/2026-10-19-17-45-29.png)

```python
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd,head_size,bias = False)
        self.query = nn.Linear(n_embd,head_size,bias = False)
        self.value = nn.Linear(n_embd,head_size,bias = False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
```
![](images/2026-10-20-14-52-16.png)
![](images/2026-10-20-15-02-52.png)
![](images/2026-10-20-15-04-29.png)
![](images/2026-10-20-15-05-21.png)
![](images/2026-10-20-15-05-35.png)
![](images/2026-10-20-15-05-56.png)
![](images/2026-10-20-15-06-06.png)
![](images/2026-10-20-15-06-13.png)
![](images/2026-10-20-15-09-57.png)

```python
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd,vocab_size)

    def forward(self, idx, targets=None):

        B,T = idx.shape
         # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)   # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T,device = device))  #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        x= self.sa_head(x)  #apply one head of self-attention. (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
```
![](images/2026-10-20-15-35-12.png)
![](images/2026-10-20-15-59-16.png)
![](images/2026-10-20-16-01-33.png)
![](images/2026-10-21-10-30-24.png)

```python
 def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:,-block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
```
![](images/2026-10-20-16-23-27.png)
![](images/2026-10-20-16-22-52.png)
![](images/2026-10-20-16-25-39.png)
![](images/2026-10-20-16-25-51.png)
![](images/2026-10-20-16-31-42.png)
![](images/2026-10-20-16-31-55.png)

```python
class MultiHeadAttention(nn.Module):
    '''multiple heads of self-attention in parallel'''
    
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd,n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads],dim = -1)
        out = self.proj(out)
        return out
```
![](images/2026-10-20-22-29-32.png)
![](images/2026-10-20-22-30-47.png)
![](images/2026-10-20-23-04-16.png)
![](images/2026-10-20-23-04-29.png)
![](images/2026-10-20-22-32-29.png)
![](images/2026-10-20-22-32-44.png)
![](images/2026-10-21-14-54-48.png)

```python
class FeedForward(nn.Module):
    '''a simple linear layer followed by a non-linearity'''
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd),
        )
    def forward(self,x):
        return self.net(x)
```
![](images/2026-10-21-10-11-14.png)
![](images/2026-10-21-10-11-38.png)
![](images/2026-10-21-10-14-08.png)
![](images/2026-10-21-14-56-10.png)
![](images/2026-10-21-14-56-24.png)

```python
class Block(nn.Module):
    '''Transformer block:communication followed by computation'''
    def __init__(self,n_embd,n_head):
        #n_embd:embedding dimension,n_head:the number of heads we'd like
        super().__init__()
        head_size = n_embd//n_head
        self.sa = MultiHeadAttention(n_head,head_size)
        self.ffwd = FeedForward(n_embd)
    def forward(self,x):
        x = self.sa(x)
        x = self.ffwd(x)
        return x
```
![](images/2026-10-21-11-22-01.png)
![](images/2026-10-21-11-22-26.png)
![](images/2026-10-21-11-22-51.png)
![](images/2026-10-21-11-23-03.png)

In [ ]:
class BatchNorm1d:
    def __init__(self,dim,eps = 1e-5,momentum = 0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        #parameters(trained with backprop)通过反向传播进行训练/更新
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        #buffers(trained with a running 'momentum update')
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):
        #calculate the forward pass
        if self.training:
            xmean = x.mean(0,keepdim = True) #batch mean
            xvar = x.var(0,keepdim = True) #batch variance
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x-xmean)/torch.sqrt(xvar+self.eps)
        self.out = self.gamma * xhat + self.beta
        #update the buffers
        if self.training:
            with torch.no_grad():
                self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*xmean
                self.running_var = (1-self.momentum)*self.running_var + self.momentum*xvar
        return self.out

    def parameters(self):
        return [self.gamma,self.beta]
    
torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32,100)     #batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(),x[:,0].std()  #BatchNorm是对行归一化！！！

(tensor(7.4506e-09), tensor(1.0000))

In [15]:
class LayerNorm:
    def __init__(self,dim,eps = 1e-5):
        self.eps = eps
        #parameters(trained with backprop)通过反向传播进行训练/更新
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
    def __call__(self,x):
        #calculate the forward pass
        xmean = x.mean(1,keepdim = True) #batch mean
        xvar = x.var(1,keepdim = True) #batch variance
        xhat = (x-xmean)/torch.sqrt(xvar+self.eps)
        out = self.gamma * xhat + self.beta
        return out

    def parameters(self):
        return [self.gamma,self.beta]
    
torch.manual_seed(1337)
module = LayerNorm(100)
x = torch.randn(32,100)
x = module(x)
x.shape

torch.Size([32, 100])

In [16]:
x[0,:].mean(),x[0,:].std()

(tensor(-9.5367e-09), tensor(1.0000))

![](images/2026-10-22-22-49-09.png)
![](images/2026-10-22-22-53-06.png)
![](images/2026-10-22-22-53-22.png)
![](images/2026-10-22-22-53-44.png)
![](images/2026-10-22-22-53-58.png)

```python
 self.blocks = nn.Sequential(*[Block(n_embd,n_head = n_head) for _ in range(n_layer)])
 self.ln_f = nn.LayerNorm(n_embd)
```
![](images/2026-10-22-23-38-01.png)
![](images/2026-10-22-23-38-55.png)
![](images/2026-10-22-23-39-10.png)
![](images/2026-10-23-10-36-02.png)
![](images/2026-10-23-10-36-44.png)
![](images/2026-10-23-10-50-55.png)
![](images/2026-10-23-10-50-04.png)

![](images/2026-10-23-15-02-17.png)
![](images/2026-10-23-15-02-36.png)
![](images/2026-10-23-15-03-08.png)
![](images/2026-10-23-15-03-54.png)

![](images/2026-10-23-15-43-20.png)
![](images/2026-10-23-15-43-33.png)
![](images/2026-10-23-15-44-42.png)

# py文件和ipynb文件运行细节
![](images/2026-10-20-16-51-36.png)
![](images/2026-10-20-16-52-06.png)
![](images/2026-10-20-16-52-28.png)
![](images/2026-10-20-16-52-38.png)